In [ ]:
import blocks as layers
import torch
import torch.nn as nn
import math

/home/cybertesla/Transformers/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
class Transformer(nn.Module):

    def __init__(self, src_vocab_size: int, tgt_vocab_size: int, d_model: int, num_heads: int, hidden_dim: int, num_blocks: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.dropout_rate = dropout_rate
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = layers.PositionalEncoding()
        self.dropout = nn.Dropout(p=dropout_rate)
        self.encoder = layers.EncoderStack(src_vocab_size, d_model, num_heads, hidden_dim, num_blocks, dropout_rate)
        self.decoder = layers.DecoderStack(tgt_vocab_size, d_model, num_heads, hidden_dim, num_blocks, dropout_rate)
        self.final_linear_layer = nn.Linear(d_model, tgt_vocab_size)


    def forward(self, src_tokens: torch.Tensor, tgt_tokens: torch.Tensor):
        # src_tokens, tgt_tokens Dimension: (Batch, Tokens)
        
        encoder_embeddings = self.encoder_embedding(src_tokens) * math.sqrt(self.d_model)
        decoder_embeddings = self.decoder_embedding(tgt_tokens) * math.sqrt(self.d_model)
        # Dimension: (Batch_size, num_tokens, d_model)
        
        encoder_positional_embeddings = self.dropout(self.positional_encoding(encoder_embeddings))
        decoder_positional_embeddings = self.dropout(self.positional_encoding(decoder_embeddings))
        # Dimension: (Batch_size, num_tokens, d_model)

        # Passing through Encoder
        encoder_output = self.encoder(encoder_positional_embeddings)

        # Passing through Decoder
        decoder_output = self.decoder(decoder_positional_embeddings, encoder_output)

        # Final Linear Layer
        logits = self.final_linear_layer(decoder_output)

        return logits

In [3]:
batch_size = 2
num_src_tokens = 3
num_tgt_tokens = 5
d_model = 8
num_heads = 2
head_dim = 4
src_vocab_size = 5
tgt_vocab_size = 7
hidden_dim = 16
num_blocks = 6

In [4]:
dummy_src_input = torch.randint(low=0, high=src_vocab_size, size=(batch_size, num_src_tokens))
dummy_tgt_input = torch.randint(low=0, high=tgt_vocab_size, size=(batch_size, num_tgt_tokens))
print(dummy_src_input.shape)
print(dummy_tgt_input.shape)
print("(batch_size, num_tokens)")

torch.Size([2, 3])
torch.Size([2, 5])
(batch_size, num_tokens)


In [5]:
transformer =Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, hidden_dim, num_blocks)
transformer

Transformer(
  (encoder): EncoderStack(
    (embedding): Embedding(5, 8)
    (positional_encoding): PositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (attention_layer): MultiHeadAttention(
          (w_q): Linear(in_features=8, out_features=8, bias=False)
          (w_k): Linear(in_features=8, out_features=8, bias=False)
          (w_v): Linear(in_features=8, out_features=8, bias=False)
          (w_o): Linear(in_features=8, out_features=8, bias=False)
        )
        (ffnn): FFNN(
          (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
          (output_layer): Linear(in_features=16, out_features=8, bias=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (norm_1): LayerNorm((8,), 

In [8]:
transformer_output = transformer(dummy_src_input, dummy_tgt_input)
print(transformer_output.shape)
print("(batch_size, num_tgt_tokens, tgt_vocab_size)")
print(transformer_output)

torch.Size([2, 5, 7])
(batch_size, num_tgt_tokens, tgt_vocab_size)
tensor([[[-5.7344e-01,  2.3760e-01,  5.2905e-01, -7.1201e-01, -5.8375e-01,
           1.7700e-01, -3.3187e-01],
         [-4.3265e-01,  3.8451e-01,  5.0630e-01, -4.3897e-01, -8.1565e-01,
           4.2386e-01, -2.9379e-01],
         [ 6.9196e-01,  3.9130e-01, -8.6248e-01, -1.8870e-01, -7.5468e-01,
           3.5278e-01,  7.4309e-01],
         [-2.9176e-01,  4.7384e-01, -6.1988e-04, -9.7680e-01,  9.9850e-01,
          -5.4507e-01,  2.6205e-01],
         [ 9.6748e-02,  4.3242e-01, -4.3970e-02,  3.4776e-01, -1.4428e+00,
           8.4246e-01, -3.4614e-01]],

        [[-4.0562e-01,  4.2904e-02,  4.5200e-01, -3.1259e-01, -7.1430e-01,
           5.4099e-01, -2.1718e-01],
         [-6.2652e-01,  5.0008e-01, -9.9226e-04, -5.4739e-02,  4.6675e-01,
          -2.2093e-01, -2.2319e-01],
         [-2.8736e-01,  3.4932e-01,  3.5655e-01,  9.5435e-02, -1.0649e+00,
           7.9932e-01, -3.3620e-01],
         [-4.8038e-02,  6.9623e-01,